In [1]:
print("hi")

hi


In [1]:
import os
import re
import json
import pdfplumber
from transformers import pipeline
from PIL import Image
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


COLUMN_HEADERS = [
    "service_code",
    "service_dates",
    "service_description",
    "amount_billed",
    "discounts_and_reductions",
    "allowed_amount",
    "dental_plan_responsibility",
    "deductible_amount",
    "copay_amount",
    "coinsurance",
    "amount_not_covered",
    "patient_costs"
]

SCHEMA = {
    "rows": [
        {
            "service_code": "",
            "service_dates": "",
            "service_description": "",
            "amount_billed": "",
            "discounts_and_reductions": "",
            "allowed_amount": "",
            "dental_plan_responsibility": "",
            "deductible_amount": "",
            "copay_amount": "",
            "coinsurance": "",
            "amount_not_covered": "",
            "patient_costs": ""
        }
    ],
    "column_totals": {
        "amount_billed": "",
        "discounts_and_reductions": "",
        "allowed_amount": "",
        "dental_plan_responsibility": "",
        "deductible_amount": "",
        "copay_amount": "",
        "coinsurance": "",
        "amount_not_covered": "",
        "patient_costs": ""
    }
}

# =========================================================
# EXTRACTION PROMPT
# =========================================================
PROMPT = """You are extracting structured financial data from a cropped EOB table image.

STRICT EXTRACTION RULES:

1. ROW HANDLING

- DO NOT HALLUCINATE ADDITIONAL ROWS.
 Preserve ALL visible service rows exactly once.
- Never repeat rows
- Never create extra rows
- Never infer missing rows

- The number of extracted service rows must exactly match the visible service rows in the table.
- Do not restart extraction from the beginning.
- Stop immediately after the final service row before CLAIM TOTALS.

Stop extracting service rows immediately when:
- "CLAIM TOTALS"
- "Column Totals"
- totals row
is reached.
- A valid service row MUST contain a service_code.
- Do NOT create rows from description-only text lines.
- Ignore wrapped description lines below a service row.

2. SERVICE DESCRIPTION as service_code
- Format: starts with "D" followed by exactly 4 digits (example: D4910).
- If no valid code found, return "".
2(a). Extract the exact Subscriber name/Patient name in the table
- Return only the patient name
 
3. SERVICE DATES
- Extract ONLY the date value (e.g., 02/27/26).
- Ignore any extra text.

4. NUMERIC FIELDS (IMPORTANT)
For the following columns, extract ONLY numeric values:

- amount_billed
- discounts_and_reductions
- allowed_amount
- dental_plan_responsibility
- deductible_amount
- copay_amount
- coinsurance
- amount_not_covered
- patient_costs

Rules:
- Remove "$", commas, and spaces.
- Convert to standard decimal format (e.g., 209.00).
- If empty → return "".
- If value contains text like "(PD)" → IGNORE that text and extract only the number.
- If cell contains only "(PD)" or non-numeric → return "".

5. COLUMN-SPECIFIC RULES
- discounts_and_reductions → extract only from that column (do not mix with others)
- amount_not_covered → extract ONLY numeric value, ignore anything in brackets like "(PD)"

6. TOTALS EXTRACTION
- Extract totals ONLY from the row labeled "CLAIM TOTALS".
- Do NOT compute totals yourself.
- Map each total strictly to its respective column:
    - amount_billed → total under Amount Billed
    - discounts_and_reductions → total under Discounts
    - allowed_amount → total under Allowed
    - dental_plan_responsibility → total under Dental Plan Responsibility
    - deductible_amount → total under Deductible
    - copay_amount → total under Copay
    - coinsurance → total under Coinsurance
    - amount_not_covered → total under Amount Not Covered
    - patient_costs → total under Patient Costs

7. STRICT COLUMN ALIGNMENT
- Do NOT shift values between columns.
- Each value must come ONLY from its column position.


9. DATA INTEGRITY (CRITICAL)
- Do NOT hallucinate values.
- Do NOT infer missing numbers.
- Do NOT modify numeric values.
- Extract exactly as seen.

10. PRIORITY
Accuracy > completeness.
If uncertain, return "" instead of guessing.

11. Must obey the output format.

12. For every extracted field, return:
   - value
   - confidence

13. Confidence must be a number between 0 and 1.

14. Confidence should represent how certain you are that the value was correctly
    read from the image.

15. If a field is not visible or cannot be reliably extracted:
    - value = ""
    - confidence = 0.0

OUTPUT FORMAT

{
    "patient_name": {
        "value": "",
        "confidence": 0.0
    },

    "rows": [
        {
            "service_code": {
                "value": "",
                "confidence": 0.0
            },

            "service_dates": {
                "value": "",
                "confidence": 0.0
            },

            "service_description": {
                "value": "",
                "confidence": 0.0
            },

            "amount_billed": {
                "value": "",
                "confidence": 0.0
            },

            "discounts_and_reductions": {
                "value": "",
                "confidence": 0.0
            },

            "allowed_amount": {
                "value": "",
                "confidence": 0.0
            },

            "dental_plan_responsibility": {
                "value": "",
                "confidence": 0.0
            },

            "deductible_amount": {
                "value": "",
                "confidence": 0.0
            },

            "copay_amount": {
                "value": "",
                "confidence": 0.0
            },

            "coinsurance": {
                "value": "",
                "confidence": 0.0
            },

            "amount_not_covered": {
                "value": "",
                "confidence": 0.0
            },

            "patient_costs": {
                "value": "",
                "confidence": 0.0
            }
        }
    ],

    "column_totals": {
        "amount_billed": {
            "value": "",
            "confidence": 0.0
        },

        "discounts_and_reductions": {
            "value": "",
            "confidence": 0.0
        },

        "allowed_amount": {
            "value": "",
            "confidence": 0.0
        },

        "dental_plan_responsibility": {
            "value": "",
            "confidence": 0.0
        },

        "deductible_amount": {
            "value": "",
            "confidence": 0.0
        },

        "copay_amount": {
            "value": "",
            "confidence": 0.0
        },

        "coinsurance": {
            "value": "",
            "confidence": 0.0
        },

        "amount_not_covered": {
            "value": "",
            "confidence": 0.0
        },

        "patient_costs": {
            "value": "",
            "confidence": 0.0
        }
    }
}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

"""
#_________________________________________________________________________________________________________________
# =========================================================
# IMAGE -> JSON EXTRACTION
# =========================================================

def extract_table_from_image(image_path):

    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": PROMPT
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=5000,
        temperature = 0.0,
        repetition_penalty = 1.2
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()

    # REMOVE MARKDOWN IF MODEL RETURNS IT
    generated_text = generated_text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    return generated_text


def enforce_schema(parsed_output):

    allowed_columns = COLUMN_HEADERS  # ✅ ordered list

    for table in parsed_output.get("tables", []):

        # =========================
        # 🔹 CLEAN ROWS (STRICT + ORDERED)
        # =========================
        cleaned_rows = []

        for row in table.get("rows", []):

            cleaned_row = {}

            for col in allowed_columns:
                cleaned_row[col] = row.get(col, "")

            service_code = cleaned_row.get(
                "service_code",
                ""
            ).strip()

            if not re.match(r"^D\d{4}$", service_code):
                continue

            cleaned_rows.append(cleaned_row)
        table["rows"] = cleaned_rows

        # =========================
        # 🔹 CLEAN COLUMN TOTALS (STRICT + ORDERED)
        # =========================
        totals = table.get("column_totals", {})
        ordered_totals = {}

        for col in allowed_columns:
            if col not in ["service_dates", "service_code"]:
                ordered_totals[col] = totals.get(col, "")

        table["column_totals"] = ordered_totals

    return parsed_output
#remove_service_description

def remove_service_description(data):
    """
    Remove service_description from final output.
    """

    if "rows" not in data:
        return data

    for service in data["rows"]:
        service.pop("service_description", None)

    return data
# =========================================================
# MAIN PIPELINE
# =========================================================
def check_claim_denied(page):

    """
    Logic:
    Search ONLY before:
    'Attention Non-contracted Medicare Providers'

    If denied / denial keywords exist before that section,
    return True else False
    """

    full_text = page.extract_text()

    if not full_text:
        return False

    # -----------------------------------------------------
    # LIMIT SEARCH AREA
    # -----------------------------------------------------

    stop_keyword = "For Claim Submissions and ReSubmissions:"

    if stop_keyword in full_text:

        full_text = full_text.split(stop_keyword)[0]

    searchable_text = full_text.lower()

    # -----------------------------------------------------
    # DENIAL KEYWORDS
    # -----------------------------------------------------

    denial_keywords = [

        "denied",
        "denial",
    ]

    # -----------------------------------------------------
    # SEARCH
    # -----------------------------------------------------

    for keyword in denial_keywords:

        if keyword in searchable_text:

            print(f"❌ Claim Denied Keyword Found: {keyword}")

            return "denied"

    return "not denied"

def crop_all_eob_tables(
    pdf_path,
    output_dir="EOB_OUTPUT/Dearborn_nation", company_name = "Blue Cross"):
    pdf_file = os.path.basename(pdf_path)
    pdf_name = pdf_file.split("_")[-1].split(".")[0]


    pdf_dir = os.path.basename(
        pdf_path
    ).split(".")[0].split("_")[-1]

    pdf_full_name = os.path.basename(pdf_path)


    output_dir = os.path.join(
        output_dir,
        pdf_dir
    )

    os.makedirs(output_dir, exist_ok=True)

    all_patients = []
    confidence_results = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(
            pdf.pages,
            start=1
        ):
            is_denied = check_claim_denied(page)

            print(f"Denied Status : {is_denied}")

            print(f"\n==================================================")
            print(f"🚀 Processing Page {page_num}")
            print(f"==================================================")

            start_positions = []
            end_positions = []

            # ============================================
            # FIND TABLE STARTS
            # ============================================

            start_hits = page.search(
                "Subscriber Name"
            )

            if not start_hits:

                start_hits = page.search(
                    "Subscriber"
                )

            # ============================================
            # FIND TABLE ENDS
            # ============================================

            end_hits = page.search(
                "TOTALS"
            )

            if not end_hits:

                end_hits = page.search(
                    "CLAIM"
                )

            # ============================================
            # EXTRACT Y POSITIONS
            # ============================================

            for hit in start_hits:

                start_positions.append(
                    hit["top"] - 1
                )

            for hit in end_hits:

                end_positions.append(
                    hit["bottom"] + 2
                )

            # ============================================
            # VALIDATION
            # ============================================

            table_count = min(
                len(start_positions),
                len(end_positions)
            )

            if table_count == 0:
                continue

            # ============================================
            # CROP TABLES
            # ============================================

            for idx in range(table_count):
                print(f"\n🧾 Processing Table {idx + 1}")

                start_y = start_positions[idx]
                end_y = end_positions[idx]

                bbox = (
                    0,
                    start_y,
                    page.width,
                    end_y
                )

                cropped_page = page.crop(
                    bbox
                )

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )

                cropped_page.to_image(
                    resolution=300
                ).save(image_path)

                try:

                    llm_output = extract_table_from_image(image_path)
                    # final_output = remove_service_description(llm_output)
                    print("\n================ RAW MODEL OUTPUT ================\n")
                    print(llm_output)
                    print("\n==================================================\n")
                    # print(final_output)

                    parsed_output = json.loads(llm_output)

                    parsed_output = {
                        "tables": [parsed_output]
                    }

                    # =========================================================
                    # CONFIDENCE CALCULATION + UNWRAP
                    # =========================================================

                    for i, table in enumerate(parsed_output.get("tables", [])):

                        # 1. Calculate confidence while
                        #    value + confidence still exist
                        table_model_confidence = calculate_model_confidence(table)

                        # 2. Remove confidence wrappers
                        #    {"value": "D9310", "confidence": 0.99}
                        #    becomes "D9310"
                        table = _unwrap_vlm_output(table)

                        # 3. Keep only the overall model confidence internally
                        table["_model_confidence"] = table_model_confidence

                        parsed_output["tables"][i] = table

                        # 4. Keep for final EOB confidence calculation
                        # confidence_results.append(table)


                    # =========================================================
                    # NOW RUN NORMAL SCHEMA VALIDATION
                    # =========================================================

                    parsed_output = enforce_schema(parsed_output)


                    # for table in parsed_output.get("tables", []):

                    #     new_table = {
                    #         "EOB_ID": pdf_name,
                    #         "patient_name": patient_name,
                    #         "rows": table.get("rows", []),
                    #         "column_totals": table.get("column_totals", {})
                    #     }

                    #     table.clear()
                    #     table.update(new_table)

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        expected_rows = count_service_rows(
                        page,
                        start_y,
                        end_y
                    )

                        actual_rows = len(
                            table.get("rows", [])
                        )

                        row_count_ok = (
                            expected_rows == actual_rows
                    )

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        
                        is_valid, log, errors, total_fields= validate_eob_table(table, t_idx)
                        if not row_count_ok:
                            is_valid = False
                            errors = errors + [{"type": "row_count_mismatch"}]

                    for table in parsed_output.get("tables", []):

                        structured_table = {
                            "EOB_ID": pdf_name,
                            "claim_status": is_denied,                           
                            "patient_name": table.get("patient_name", ""),
                            "rows": table.get("rows", []),
                            "totals": table.get("column_totals", {}),
                            "validation": {"status": is_valid, "errors": errors},
                            "_expected_rows": expected_rows,
                            "_total_fields": total_fields,

                        }

                        print(f"✅ Completed Table {idx + 1} on Page {page_num}")
                        print(all_patients)

                    date_of_service = ""

                    if structured_table.get("rows"):
                        date_of_service = structured_table["rows"][0].get("service_dates", "")

                    for row in structured_table.get("rows", []):

                        for col in ["service_description", "service_dates"]:
                            row.pop(col, None)

                    structured_table["totals"].pop("service_description", None)

                    patient_data = {
                        # "EOB_id": structured_table.get("EOB_ID", ""),
                        "patient_name": structured_table.get("patient_name", ""),
                        "date_of_service": date_of_service,
                        "services": structured_table.get("rows", []),
                        "totals": structured_table.get("totals", {}),
                        "validation": structured_table.get("validation"),
                        "_expected_rows": expected_rows,
                        "_total_fields": total_fields,
                        "_model_confidence": table.get("_model_confidence", 0.0),
                    }

                    all_patients.append(patient_data)
                    confidence_results.append(patient_data)

                    # for row in structured_table.get("rows", []):

                    #     row.pop(
                    #         "service_description",
                    #         None
                    #     )                   

                    #     structured_table["totals"].pop(
                    #         "service_description",
                    #         None
                    #     )                    

                    # results.append(structured_table)

                except Exception as e:

                    all_patients.append({
                        "page": page_num,
                        "table": idx + 1,
                        "status": "failed",
                        "image_path": image_path,
                        "error": str(e)
                    })

    confidence_score = calculate_eob_confidence(confidence_results)


    final_output=[
        {
        "eob_id": pdf_name,
        "file_name":pdf_full_name,
        "payor": "Blue Cross and Blue Shield of Texas",
        "claim_status": is_denied,
        "confidence_score": confidence_score,
        "patients": all_patients
    }
]                  



    success_path, failed_path = save_split_output(
                final_output,
                company_name=company_name,
                pdf_name=pdf_name,
                pdf_path=pdf_path,
                cropped_dir=output_dir,
            )
        
    print(f"\n📁 Cropped images : {output_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output

#------------------- PARSE AMOUNT --------------------#

def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())

#------------------- VALIDATE TOTAL AMOUNT --------------------#

import re

def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()
    row_positions = []

    for w in words:
        text = w["text"].strip()

        match = re.search(r"\bD\d{4}\b", text)

        if match:
            y = float(w["top"])

            if region_top <= y <= region_bottom:
                row_positions.append(y)

    row_positions.sort()

    grouped_rows = []
    threshold = 3

    for y in row_positions:
        if not grouped_rows:
            grouped_rows.append(y)
        else:
            if abs(y - grouped_rows[-1]) > threshold:
                grouped_rows.append(y)

    return len(grouped_rows)


def validate_eob_table(table: dict, table_index: int):

    rows = table.get("rows", [])
    totals = table.get("column_totals", {})

    if not rows:
        return False, "", [], 0

    computed_totals = {
        "amount_billed": round(sum(parse_amount(r.get("amount_billed", "")) for r in rows), 2),
        "discounts_and_reductions": round(sum(parse_amount(r.get("discounts_and_reductions", "")) for r in rows), 2),
        "allowed_amount": round(sum(parse_amount(r.get("allowed_amount", "")) for r in rows), 2),
        "dental_plan_responsibility": round(sum(parse_amount(r.get("dental_plan_responsibility", "")) for r in rows), 2),
        "deductible_amount": round(sum(parse_amount(r.get("deductible_amount", "")) for r in rows), 2),
        "copay_amount": round(sum(parse_amount(r.get("copay_amount", "")) for r in rows), 2),
        "coinsurance": round(sum(parse_amount(r.get("coinsurance", "")) for r in rows), 2),
        "amount_not_covered": round(sum(parse_amount(r.get("amount_not_covered", "")) for r in rows), 2),
        "patient_costs": round(sum(parse_amount(r.get("patient_costs", "")) for r in rows), 2)
    }

    result_validation = ""
    errors = []
    has_error = False

    total_fields = len(computed_totals)

    print(f"\n🔍 Validation for [Table {table_index}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    if has_error:
        print(f"❌ [Table {table_index}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [Table {table_index}] Validation PASSED\n")
        return True, result_validation, [], total_fields

def validate_service_row_count(page, start_y, end_y, table, table_index):

    detected_count = count_service_rows(page, start_y, end_y)

    rows = table.get("rows", [])
    extracted_count = len([
        r for r in rows
        if r.get("service_code") not in ["", None]
    ])

    print(f"\n📊 Row Count Validation [Table {table_index}]")
    print("-" * 70)

    if detected_count == extracted_count:
        icon = "✅"
        status = "match"
    else:
        icon = "❌"
        status = "MISMATCH"

    print(f"{icon} row_count detected={detected_count:<5} | extracted={extracted_count:<5} {status}")
    print("-" * 70)


    return detected_count == extracted_count

W0901 18:43:19.162000 3457835 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 18:43:19.177000 3457835 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [3]:
import os

folder_path = r"/home/cipl/users/OCR_Project/DEARBORN_NATIONAL/Dearborn_PDF"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            crop_all_eob_tables(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_846828828.pdf
Denied Status : not denied

🚀 Processing Page 1

🧾 Processing Table 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`



================ RAW MODEL OUTPUT ================

{
    "patient_name": {
        "value": "OSIYEMI,OLUMADE S",
        "confidence": 1.0
    },

    "rows": [
        {
            "service_code": {
                "value": "D0274",
                "confidence": 1.0
            },

            "service_dates": {
                "value": "03/10/26",
                "confidence": 1.0
            },

            "service_description": {
                "value": "Bitewings - Four Radiographic Images",
                "confidence": 1.0
            },

            "amount_billed": {
                "value": "104.00",
                "confidence": 1.0
            },

            "discounts_and_reductions": {
                "value": "37.00",
                "confidence": 1.0
            },

            "allowed_amount": {
                "value": "67.00",
                "confidence": 1.0
            },

            "dental_plan_responsibility": {
                "value": "67.00",
       

[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



================ RAW MODEL OUTPUT ================

{
    "patient_name": {
        "value": "TAYLOR,ROBERT E",
        "confidence": 1.0
    },

    "rows": [
        {
            "service_code": {
                "value": "D4910",
                "confidence": 1.0
            },

            "service_dates": {
                "value": "02/27/26",
                "confidence": 1.0
            },

            "service_description": {
                "value": "Periodontal Maintenance",
                "confidence": 1.0
            },

            "amount_billed": {
                "value": "209.00",
                "confidence": 1.0
            },

            "discounts_and_reductions": {
                "value": "68.00",
                "confidence": 1.0
            },

            "allowed_amount": {
                "value": "141.00",
                "confidence": 1.0
            },

            "dental_plan_responsibility": {
                "value": "72.80",
                "conf

...


================ RAW MODEL OUTPUT ================

{
    "patient_name": {
        "value": "PETRONIO,LAYLA",
        "confidence": 1.0
    },

    "rows": [
        {
            "service_code": {
                "value": "D0120",
                "confidence": 1.0
            },

            "service_dates": {
                "value": "07/14/23",
                "confidence": 1.0
            },

            "service_description": {
                "value": "Periodic Oral Evaluation - Established P",
                "confidence": 1.0
            },

            "amount_billed": {
                "value": "53.00",
                "confidence": 1.0
            },

            "discounts_and_reductions": {
                "value": "24.00",
                "confidence": 1.0
            },

            "allowed_amount": {
                "value": "29.00",
                "confidence": 1.0
            },

            "dental_plan_responsibility": {
                "value": "29.00",
       

[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`



================ RAW MODEL OUTPUT ================

{
    "patient_name": {
        "value": "HALL,PAMELA",
        "confidence": 1.0
    },

    "rows": [
        {
            "service_code": {
                "value": "D0210",
                "confidence": 1.0
            },

            "service_dates": {
                "value": "06/01/23",
                "confidence": 1.0
            },

            "service_description": {
                "value": "Intraoral - Comprehensive Series Of Radi",
                "confidence": 1.0
            },

            "amount_billed": {
                "value": "169.13",
                "confidence": 1.0
            },

            "discounts_and_reductions": {
                "value": "82.13",
                "confidence": 1.0
            },

            "allowed_amount": {
                "value": "87.00",
                "confidence": 1.0
            },

            "dental_plan_responsibility": {
                "value": "87.00",
         

[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



================ RAW MODEL OUTPUT ================

{
    "patient_name": {
        "value": "HEYWARD,FRANCIS",
        "confidence": 1.0
    },
    "rows": [
        {
            "service_code": {
                "value": "D0140",
                "confidence": 1.0
            },
            "service_dates": {
                "value": "05/15/23",
                "confidence": 1.0
            },
            "service_description": {
                "value": "Limited Oral Evaluation - Prob",
                "confidence": 1.0
            },
            "amount_billed": {
                "value": "58.00",
                "confidence": 1.0
            },
            "discounts_and_reductions": {
                "value": "13.00",
                "confidence": 1.0
            },
            "allowed_amount": {
                "value": "45.00",
                "confidence": 1.0
            },
            "dental_plan_responsibility": {
                "value": "45.00",
                "confid

In [3]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/DEARBORN_NATIONAL/Dearborn_PDF/Pmt_EOP_356069614.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Denied Status : not denied

🚀 Processing Page 1

🧾 Processing Table 1


[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



================ RAW MODEL OUTPUT ================

{
    "patient_name": {
        "value": "HEYWARD,FRANCIS",
        "confidence": 1.0
    },

    "rows": [
        {
            "service_code": {
                "value": "D0140",
                "confidence": 1.0
            },

            "service_dates": {
                "value": "05/15/23",
                "confidence": 1.0
            },

            "service_description": {
                "value": "Limited Oral Evaluation - Prob",
                "confidence": 1.0
            },

            "amount_billed": {
                "value": "58.00",
                "confidence": 1.0
            },

            "discounts_and_reductions": {
                "value": "13.00",
                "confidence": 1.0
            },

            "allowed_amount": {
                "value": "45.00",
                "confidence": 1.0
            },

            "dental_plan_responsibility": {
                "value": "45.00",
                

[{'eob_id': '356069614',
  'payor': 'Blue Cross and Blue Shield of Texas',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'HEYWARD,FRANCIS',
    'date_of_service': '05/15/23',
    'services': [{'service_code': 'D0140',
      'amount_billed': '58.00',
      'discounts_and_reductions': '13.00',
      'allowed_amount': '45.00',
      'dental_plan_responsibility': '45.00',
      'deductible_amount': '0.00',
      'copay_amount': '0.00',
      'coinsurance': '0.00',
      'amount_not_covered': '0.00',
      'patient_costs': '0.00'},
     {'service_code': 'D0220',
      'amount_billed': '17.00',
      'discounts_and_reductions': '0.00',
      'allowed_amount': '17.00',
      'dental_plan_responsibility': '17.00',
      'deductible_amount': '0.00',
      'copay_amount': '0.00',
      'coinsurance': '0.00',
      'amount_not_covered': '0.00',
      'patient_costs': '0.00'},
     {'service_code': 'D0230',
      'amount_billed': '13.00',
      'disco